<a href="https://colab.research.google.com/github/lversen/PHIS/blob/docker-compose-official/2026_PheNO_PHIS_DataImport_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src='https://drive.google.com/uc?export=view&id=1jyAgAg3L5dUSfO9zfp_bXVZrlV8HM-Z9'>




# Import library and Set Working Environment

## Install OpenSilex in the current Environment
OpenSilex GitHub Page referencing Python package installation, and full Documentation for API Endpoints: </Br>
https://github.com/OpenSILEX/opensilexClientToolsPython </Br>
https://opensilex-scripts.pages.mia.inra.fr/opensilex-generic-scripts/scripts/overview

In [ ]:
!pip install git+https://github.com/OpenSILEX/opensilexClientToolsPython.git@1.3.3

## Import all requiered library

In [ ]:
### Import library ##########################################################################################################
import os
import opensilexClientToolsPython as osC
import pandas as pd
import sys
import datetime
import time
from lxml import etree as ET
import json
from tqdm import tqdm
import yaml
from google.colab import data_table

print("Library Imported")

## Setting Up the Working Environment

Clone all data related to this exercise from the NaPPI-specific GitHub repository and Set the working directory.
<img src='https://drive.google.com/uc?export=view&id=1Dk4n2ncgkPHUnjCuJZh36vHuhWm7Fcwt'>

In [ ]:
### Clone all Data from GitHub ##############################################################################################
!git clone https://github.com/NaPPI-T/2026_PheNO-Workshop_PHIS

### Set Database wd #########################################################################################################
wd = '/content/2026_PheNO-Workshop_PHIS'

# Connect to the PHIS Demo instance
https://phis.emphasis.fedcloud.eu/egi-demo/app/</Br>
Connect as guest


In [ ]:
### Get Login Info ##########################################################################################################
with open (os.path.join(wd, '00-yaml', 'Demo-Guest-login.yaml'), 'r') as stream:
    login=yaml.safe_load(stream)

### Connect to PHIS Database ################################################################################################
Py_Client = osC.ApiClient()
Py_Client.connect_to_opensilex_ws(identifier=login["Identifier"],
                                  password=login["Password"],
                                  host=login["Host"])
#print(Py_Client.default_headers['Authorization'])
if Py_Client.default_headers:
    print("Connected")

# Get Experiment Info
<img src='https://drive.google.com/uc?export=view&id=1dy_OQTIckBbUxPA1RndMSGR2cM8HVHAa'>


In [ ]:
prefix='SP_' #CHANGE THIS STRING TO GENERATE YOUR PERSONALIZED EXPERIMENT NAME

### Set Experiment Name #####################################################################################################
NameExp = prefix + "24BPROBARG20_TraitFinder"
print(f'Experiment Name is "{NameExp}"\n')

### Get Exp Info ############################################################################################################
with open (os.path.join(wd, '00-yaml', 'ExpInfo.yaml'), 'r') as stream:
    ExpInfo=yaml.safe_load(stream)

display(ExpInfo)

# Get or Create Experiment Name&URI

In [ ]:
### Get or Create Scientific Objects Name&URI ###############################################################################
Exp_Api = osC.ExperimentsApi(Py_Client)
Exp_Src = Exp_Api.search_experiments(name=NameExp)["result"]

NameExp_uri = {}
if Exp_Src:
    NameExp_uri.update({NameExp: Exp_Src[0].uri})
    print("Experiment URI:{}".format(NameExp_uri[NameExp]))
    del Exp_Api, Exp_Src

    # Get Facilities Name&URI ###############################################################################################
    Org_Api = osC.OrganizationsApi(Py_Client)
    Facilities = ExpInfo['facilities']
    #print("Organization: {}".format(str(Organisation)))
    Facilities_uri = {}
    if Facilities == None:
        print("Organisation Missing")
        ls_Facilities=None
    else:
      for facility in Facilities:
        Org_Src = Org_Api.search_facilities(pattern=facility)["result"]
        if Org_Src:
          Facilities_uri.update({facility: Org_Src[0].uri})
          print("{} URI: {}".format(facility, Org_Src[0].uri))
        else:
            print("\033[91m{}: Unknown Facility\033[0m".format(facility))
            del facility, Org_Src
    del Org_Api, Facilities

else:
    # List Exp Mandatory Info ###############################################################################################
    ObjectiveExp = ExpInfo['Objective']
    if ObjectiveExp != None:
      print("Objective: {}".format(ObjectiveExp))
    else:
      sys.exit("Objective Missing")

    StartExp = ExpInfo['Start Date']
    #print(ExpInfo['Start Date'])
    if StartExp != None:
      print("Start Date: {}".format(StartExp))
    else:
      sys.exit("Starting Date Missing")
    print('#'*50)

    # Get Description Name&URI ##############################################################################################
    DescriptionExp = ExpInfo['Description']
    print("Description: {}".format(str(DescriptionExp)))
    print('#'*50)

    # Get End Date Name&URI #################################################################################################
    EndExp = ExpInfo['End Date']
    #print("End Date: {}".format(EndExp))
    if EndExp != None:
      print("Start Date: {}".format(EndExp))
    else:
      print("Ending Date Missing")

    # Get Is_Public Info ####################################################################################################
    Is_Public = ExpInfo['Is_Public']
    print("Is_Public: {}".format(str(Is_Public)))
    print('#'*50)

    # Get Organisation Name&URI #############################################################################################
    Org_Api = osC.OrganizationsApi(Py_Client)
    Organisation = ExpInfo['organisations']
    #print("Organization: {}".format(str(Organisation)))
    Organisation_uri = {}
    if Organisation == None:
        print("Organisation Missing")
        ls_Organisation=None
    else:
      for organisation in Organisation:
        Org_Src = Org_Api.search_organizations(pattern=organisation)["result"]
        if Org_Src:
          Organisation_uri.update({organisation: Org_Src[0].uri})
          print("{} URI: {}".format(organisation, Org_Src[0].uri))
          ls_Organisation=list(Organisation_uri.values())
        else:
            print("\033[91m{}: Unknown Organisation\033[0m".format(organisation))
            ls_Organisation=None
        del organisation, Org_Src
    del Org_Api, Organisation
    print('#'*50)

    # Get Groups Name&URI ###################################################################################################
    Sec_Api = osC.SecurityApi(Py_Client)
    Groups = ExpInfo['groups']
    #print("Groups: {}".format(str(Groups)))
    Groups_uri = {}
    if Groups == None:
        print("Group Missing")
        ls_Groups=None
    else:
      for group in Groups:
        Sec_Src = Sec_Api.search_groups(name=group)["result"]
        if Sec_Src:
          Groups_uri.update({group: Sec_Src[0].uri})
          print("{} URI: {}".format(group, Sec_Src[0].uri))
          ls_Groups=list(Groups_uri.values())
        else:
            print("\033[91m{}: Unknown Group\033[0m".format(group))
            ls_Groups=None
        del group, Sec_Src
    del Sec_Api, Groups
    print('#'*50)

    # Get Project Name&URI ##################################################################################################
    Proj_Api = osC.ProjectsApi(Py_Client)
    Projects = ExpInfo['projects']
    #print("Projects: {}".format(str(Projects)))
    Projects_uri = {}
    if Projects == None:
        print("Project Missing")
        ls_Projects=None
    else:
        for project in Projects:
            Proj_Src = Proj_Api.search_projects(name=project)["result"]
            if Proj_Src:
                Projects_uri.update({project: Proj_Src[0].uri})
                print("{} URI: {}".format(project, Proj_Src[0].uri))
                ls_Projects=list(Projects_uri.values())
            else:
                print("\033[91m{}: Unknown Project\033[0m".format(project))
                ls_Projects=None
            del project, Proj_Src
    del Proj_Api, Projects
    print('#'*50)

    # Get Facilities Name&URI ###############################################################################################
    Org_Api = osC.OrganizationsApi(Py_Client)
    Facilities = ExpInfo['facilities']
    #print("Organization: {}".format(str(Organisation)))
    Facilities_uri = {}
    if Facilities == None:
        print("Organisation Missing")
        ls_Facilities=None
    else:
      for facility in Facilities:
        Org_Src = Org_Api.search_facilities(pattern=facility)["result"]
        if Org_Src:
          Facilities_uri.update({facility: Org_Src[0].uri})
          print("{} URI: {}".format(facility, Org_Src[0].uri))
          ls_Facilities=list(Facilities_uri.values())
        else:
            print("\033[91m{}: Unknown Facility\033[0m".format(facility))
            ls_Facilities=None
        del facility, Org_Src
    del Org_Api, Facilities
    print('#'*50)

    # Get Scientific Supervisor Name&URI ####################################################################################
    Sec_Api = osC.SecurityApi(Py_Client)
    Scientific_Supervisors = ExpInfo['scientific_supervisors']
    #print("Scientific Supervisors: {}".format(str(Scientific_Supervisors)))
    Scientific_Supervisors_uri = {}
    if Scientific_Supervisors == None:
        print("Scientific Supervisors Missing")
        ls_Scientific_Supervisors=None
    else:
      for scisup in Scientific_Supervisors:
        Sec_Src = Sec_Api.search_persons(name=scisup)["result"]
        if Sec_Src:
          Scientific_Supervisors_uri.update({scisup: Sec_Src[0].uri})
          print("{} URI: {}".format(scisup, Sec_Src[0].uri))
          ls_Scientific_Supervisors=list(Scientific_Supervisors_uri.values())
        else:
            print("\033[91m{}: Unknown Scientific Supervisors\033[0m".format(scisup))
            ls_Scientific_Supervisors=None
        del scisup, Sec_Src
    del Sec_Api, Scientific_Supervisors
    print('#'*50)

    # Get Technical Supervisor Name&URI #####################################################################################
    Sec_Api = osC.SecurityApi(Py_Client)
    Technical_Supervisors = ExpInfo['technical_supervisors']
    #print("Technical Supervisors: {}".format(str(Technical_Supervisors)))
    Technical_Supervisors_uri = {}
    if Technical_Supervisors == None:
        print("Technical Supervisors Missing")
        ls_Technical_Supervisors=None

    else:
      for techsup in Technical_Supervisors:
        Sec_Src = Sec_Api.search_persons(name=techsup)["result"] ### need to search person
        if Sec_Src:
          Technical_Supervisors_uri.update({techsup: Sec_Src[0].uri})
          print("{} URI: {}".format(techsup, Sec_Src[0].uri))
          ls_Technical_Supervisors=list(Technical_Supervisors_uri.values())

        else:
            print("\033[91m{}: Unknown Technical Supervisors\033[0m".format(techsup))
            ls_Technical_Supervisors=None
        del techsup, Sec_Src
    del Sec_Api, Technical_Supervisors
    print('#'*50)

    # Create Experiment #####################################################################################################
    body = osC.ExperimentCreationDTO(
        name=NameExp,
        start_date=StartExp,
        end_date=EndExp,
        description=DescriptionExp,
        objective=ObjectiveExp,
        organisations=ls_Organisation,
        projects=ls_Projects,
        facilities=ls_Facilities,
        scientific_supervisors=ls_Scientific_Supervisors,
        technical_supervisors=ls_Technical_Supervisors,
        groups=ls_Groups,
        is_public=Is_Public)
    Api_Resp = Exp_Api.create_experiment(body=body,)
    print("Experiment Creation: {}".format(str(Api_Resp["metadata"]["datafiles"])))

    ### Get Experiment Name&URI #############################################################################################
    Exp_Src = Exp_Api.search_experiments(name=NameExp)
    NameExp_uri.update({NameExp: Exp_Src["result"][0].uri})
    print( "{} URI: {}".format(NameExp,  NameExp_uri[NameExp]))
    del Exp_Api, body, Api_Resp, DescriptionExp, ObjectiveExp, StartExp, EndExp, Is_Public, Organisation_uri, Groups_uri, Scientific_Supervisors_uri, Technical_Supervisors_uri

# Get Numerical Data

In [ ]:
### Define the target datetime format with timezone offset ##################################################################
desired_format = "%Y-%m-%dT%H:%M:%S%z"

df_data = pd.read_excel(os.path.join(wd,'PBar1x4_TraitFinder_20260107_PHIS.xlsx'))

df_data['Plant_ID']= prefix + df_data['Plant_ID']

df_data['Timestamp'] = pd.to_datetime(df_data['Timestamp'], format="%Y-%m-%d %H:%M:%S")
df_data['Timestamp'] = df_data['Timestamp'].dt.tz_localize('Europe/Oslo').dt.strftime("%Y-%m-%dT%H:%M:%S%z")

df_data.head()

# Reference Scientific Object
Your observed object (e.g., Plant, Leaf, Plot) is linked to several metadata elements that provide essential context and description:



*   Has Germplasm: Specifies the genetic material or cultivar used in the
experiment.
*   Has Factor Level: Describes the specific treatment or environmental condition applied (e.g., drought level, fertilizer type, temperature).
*   Has Facility: Indicates the location or controlled environment where the observation was conducted.
*   ...

These metadata help ensure that the scientific object is fully described.

<img src='https://drive.google.com/uc?export=view&id=1Kp79lO6dFdZBorh_Qy1tnjsUBug4g76c'>

## Get or Create Germplasms Name&URI

In [ ]:
### Get Germplasms Info #####################################################################################################
with open (os.path.join(wd, '00-yaml', 'Germplasm.yaml'), 'r') as stream:
    Germplasms=yaml.safe_load(stream)

display(Germplasms)

In [ ]:
### Get Subtaxa Name&URI ####################################################################################################
Germ_Api = osC.GermplasmApi(Py_Client)

Species_uri={}
Germplasms_uri={}

for entry in Germplasms:
    for species, rdftype_sp in entry['Species'].items():

        Germ_Src = Germ_Api.search_germplasm(name=f"^{species}$", # Should search for exact match
                                             rdf_type=rdftype_sp)["result"]

        if Germ_Src:
            Species_uri.update({species: Germ_Src[0].uri})
            print('#'*50)
            print(f"{species}, {rdftype_sp} URI: {Species_uri[species]}")
            del Germ_Src, rdftype_sp

        else:
            check_only = False
            body = osC.GermplasmCreationDTO(
                name=species,
                rdf_type=rdftype_sp)

            Germ_Api.create_germplasm(body=body, check_only=check_only)

            Germ_Src = Germ_Api.search_germplasm(name=f"^{species}$", # Should search for exact match
                                                 rdf_type=rdftype_sp)["result"]

            Species_uri.update({species: Germ_Src[0].uri})
            print('#'*50)
            print(f"{species}, {rdftype_sp} Creation: {Germ_Src[0].uri}")
            del Germ_Src, rdftype_sp, body

        for germplasms in entry['Germplasm']:

            for germplasm, rdftype_g in germplasms.items():

                Germ_Src = Germ_Api.search_germplasm(name=f"^{germplasm}$", # Should search for exact match
                                                     rdf_type=rdftype_g)["result"]

                if Germ_Src:
                    Germplasms_uri.update({germplasm: Germ_Src[0].uri})
                    print(f"{germplasm}, {rdftype_g} URI: {Germplasms_uri[germplasm]}")
                    del Germ_Src, rdftype_g

                else:
                    check_only = False
                    body = osC.GermplasmCreationDTO(
                        name=germplasm,
                        rdf_type=rdftype_g,
                        species=Species_uri[species])

                    Germ_Api.create_germplasm(body=body, check_only=check_only)

                    Germ_Src = Germ_Api.search_germplasm(name=f"^{germplasm}$", # Should search for exact match
                                                         rdf_type=rdftype_g)["result"]

                    Germplasms_uri.update({germplasm: Germ_Src[0].uri})
                    print(f"{germplasm}, {rdftype_g} Creation: {Germ_Src[0].uri}")
                    del Germ_Src, rdftype_g, body
del species, germplasm, Germ_Api

## Get or Create Factors&Levels Names&URI
<img src='https://drive.google.com/uc?export=view&id=1aTTyerKrVJJr_7fNUa6Hu6WCGw5eDy4m'>



In [ ]:
### Get Exp Info ############################################################################################################
with open (os.path.join(wd, '00-yaml', 'Factors.yaml'), 'r') as stream:
    Factors=yaml.safe_load(stream)

display(Factors)

In [ ]:
### Search Factors ##########################################################################################################
Fac_Api = osC.FactorsApi(Py_Client)

Factors_uri = {}
Factors_Levels_uri = {}

for factor in Factors:
    Fac_Src = Fac_Api.search_factors(name=factor,
                                     experiment=NameExp_uri[NameExp])["result"]

    # Get Existing Factors Name&URI #########################################################################################
    if Fac_Src:
        Factors_uri.update({factor: Fac_Src[0].uri})
        del Fac_Src

    # Create Factors ########################################################################################################
    else:
        # Creation of LevelsDTO #############################################################################################
        lvls = {}
        for i in Factors:
            lvl = []
            for j in Factors[i]['Levels']:
                lvl.append(osC.FactorLevelCreationDTO(name=j))
            lvls[i] = lvl

        # Creation of FactorDTO #############################################################################################
        bodies = []
        for i in lvls:
            body = osC.FactorCreationDTO(name=i,
                                         levels=lvls[i],
                                         experiment=NameExp_uri[NameExp],
                                         description=Factors[i]['Description'])
            bodies.append(body)

        # Creation of Factors ###############################################################################################
        for i in bodies:
            Api_Resp = Fac_Api.create_factor(body=i,)
            print("Factors Creation: {}".format(str(Api_Resp["metadata"]["datafiles"])))
        del lvls, lvl, bodies, body, Api_Resp, Fac_Src, i, j

        # Get New Factors Name&URI ##########################################################################################
        Fac_Src = Fac_Api.search_factors(name=factor,
                                         experiment=NameExp_uri[NameExp])["result"]
        Factors_uri.update({factor: Fac_Src[0].uri})

### Get Factors Levels Name&URI #############################################################################################
for fac_uri in Factors_uri.values():
    Fac_Get = Fac_Api.get_factor_levels(uri=fac_uri)["result"]
    for lvl in Fac_Get:
        Factors_Levels_uri.update({lvl.name: lvl.uri})
del factor, fac_uri, Fac_Get, lvl

### Print Factors Name&URI ##################################################################################################
for factor in Factors_uri:
    print("{} URI: {}".format(factor, Factors_uri[factor]))
for lvl in Factors_Levels_uri:
    print("{} URI: {}".format(lvl, Factors_Levels_uri[lvl]))
del factor, lvl, Fac_Api, Factors

In [ ]:
Factors_Levels_uri=[]

## Extract Metadata

<img src='https://drive.google.com/uc?export=view&id=1bZy8hRCS2hik1icjhlC_H53V2p76TOBf'>

In [ ]:
### Get PID from DataFrame ##################################################################################################
PID = df_data['Sensor'].unique()[0]
print(f'PID found: {PID}')

### Create a new dataframe with unique rows from 'Tray ID' to 'Factor Level', dropping duplicates and NaN values ############
df_ScObj = df_data[["Block", "Column", "Row", "Plant_ID", "Germplasm"]].drop_duplicates().dropna()

# Display the first few rows of the new dataframe
df_ScObj.head()

## Get or Create Scientific Objects Name&URI
<img src='https://drive.google.com/uc?export=view&id=1Nb9NzE8TfUAbIwHifj-fwtIWbmHVfwho'>

<img src='https://drive.google.com/uc?export=view&id=1uhCIhLu9mu96e8FXhJWzTdonbV5eUW9B'>


In [ ]:
Factors_Levels_uri=[]

In [ ]:
### Get or Create Scientific Objects Name&URI ###############################################################################
Relations_Gen = []

### ObjectRealtionDTO for Start Date ########################################################################################
StartExp = ExpInfo['Start Date']
if StartExp != None:
    relation_temp = osC.RDFObjectRelationDTO(
        _property="vocabulary:hasCreationDate",
        value=StartExp)
    Relations_Gen.append(relation_temp)
    del relation_temp, StartExp

else:
    print('Start Date Missing')

EndExp = ExpInfo['End Date']
if EndExp != None:
    relation_temp = osC.RDFObjectRelationDTO(
        _property="vocabulary:hasDestructionDate",
        value=EndExp)
    Relations_Gen.append(relation_temp)
    del relation_temp, EndExp
else:
    print('End Date Missing')

### Scientific Object RDF Type ##############################################################################################
BioMat_Type = ExpInfo['RDF Type']
if not BioMat_Type:
    sys.exit("Scientific Object RDF Type Missing")
else:
    for biomat in BioMat_Type:
        Onto_Api = osC.OntologyApi(Py_Client)
        Onto_Src = Onto_Api.search_sub_classes_of(
            name=biomat, parent_type="vocabulary:ScientificObject")["result"]
        if Onto_Src:
            rdf_type = Onto_Src[0].children[0].uri
        else:
            sys.exit("Scientific Object RDF Type Unknown")
    del biomat, BioMat_Type, Onto_Api, Onto_Src

### Scientific Object Localiztion ###########################################################################################
relation_temp = osC.RDFObjectRelationDTO(
    _property="vocabulary:isHosted",
    value=Facilities_uri[ExpInfo['growth facility']])
Relations_Gen.append(relation_temp)
del relation_temp

### Get Scientific Objects Name&URI #########################################################################################
ScObj_Api = osC.ScientificObjectsApi(Py_Client)
ScObj_uri = {}
for index, row in tqdm(df_ScObj.iterrows(), desc="ScObj processing:"):
    plantid = row["Plant_ID"]
    ScObj_Src = ScObj_Api.search_scientific_objects(name=f"^{plantid}$")["result"] # Should search for exact match
    if ScObj_Src:
        ScObj_uri.update({plantid: ScObj_Src[0].uri})
    else:
        Relations_ScObj=[]

        # ObjectRealtionDTO for Germplasm ###################################################################################
        if 'Germplasms_uri' in globals() and Germplasms_uri:
            relation_temp = osC.RDFObjectRelationDTO(
                _property="vocabulary:hasGermplasm",
                value=Germplasms_uri.get(row["Germplasm"]))
            Relations_ScObj.append(relation_temp)
            del relation_temp

        # ObjectRealtionDTO for Factors #####################################################################################
        if 'Factors_Levels_uri' in globals() and Factors_Levels_uri:
            relation_temp = osC.RDFObjectRelationDTO(
                _property="vocabulary:hasFactorLevel",
                value=Factors_Levels_uri.get(row["Factor Level"]))
            Relations_ScObj.append(relation_temp)

        Relations = Relations_Gen + Relations_ScObj

        # Creation of Scientific Object #####################################################################################
        body = osC.ScientificObjectCreationDTO(name=plantid,
                                               rdf_type=rdf_type,
                                               relations=Relations,
                                               experiment=NameExp_uri[NameExp])
        ScObj_Api.create_scientific_object(body, )
        del Relations, Relations_ScObj

        # Get New Scientific Object Name&URI ################################################################################
        ScObj_Src = ScObj_Api.search_scientific_objects(name=plantid)["result"]
        ScObj_uri.update({plantid: ScObj_Src[0].uri})
print("Done")
del index, row, ScObj_Src, ScObj_Api, Relations_Gen, rdf_type

display(list(ScObj_uri.items())[0:10])

# Get Provenance Name&URI
<img src='https://drive.google.com/uc?export=view&id=1Y0vid9JzsjaCdCaU3KPrJqYgMwRsps1k'>

## Get or Create Point Cloud Provenance Name&URI

In [ ]:
### Initialize an empty dictionary to store provenance data and set the facility type #######################################
prov_dict = {}

### Get Fish-Eye Corrected Provenance #######################################################################################
Dat_Api = osC.DataApi(Py_Client)

prov = str(f"{PID}_{ExpInfo['datatype']}")

Prov_Src = Dat_Api.search_provenance(name=prov,)["result"]
if Prov_Src:
    prov_dict.update({prov: Prov_Src[0].uri})
    print("{} URI: {}".format(prov,Prov_Src[0].uri))
else:
    description = "Mesh generated from the PlantEye camera point cloud of TraitFinder."

    prov_activity = [osC.ActivityCreationDTO(rdf_type="vocabulary:Computation")]

    prov_agent = [

        osC.AgentModel(uri="phis-egi-demo:id/device/planteye_f600",
                           rdf_type="vocabulary:LaserLightSection"),

        osC.AgentModel(uri="phis-egi-demo:id/device/traitfinder",
                           rdf_type="vocabulary:HorizontalScreen"),

        osC.AgentModel(uri="phis-egi-demo:id/device/hortcontrol_314",
                           rdf_type="vocabulary:Software")
        ]

    body = osC.ProvenanceCreationDTO(name=prov,
                                     description=description,
                                     prov_agent=prov_agent,
                                     prov_activity=prov_activity)

    Api_Resp = Dat_Api.create_provenance(body=body, )
    print("Provenance Created: {}".format(str(Api_Resp["metadata"]["datafiles"])))
    Prov_Src = Dat_Api.search_provenance(name=prov,)["result"]
    prov_dict.update({prov: Prov_Src[0].uri})
    print("{} URI Created: {}".format(prov,  Prov_Src[0].uri))
    del description, prov_activity, prov_agent, body, Api_Resp
del prov, Dat_Api, Prov_Src

## Get or Create Morphological Parameters Provenance Name&URI

In [ ]:
### Get MorphoParameters Provenance #########################################################################################
Dat_Api = osC.DataApi(Py_Client)

prov = str(f"{ExpInfo['software']}_Parameters")

Prov_Src = Dat_Api.search_provenance(name=prov,)["result"]
if Prov_Src:
    prov_dict.update({prov: Prov_Src[0].uri})
    print("{} URI: {}".format(prov, Prov_Src[0].uri))
else:
    description = "Morphological and Physiological parameters computed by HortControl version 3.14"
    prov_activity = [osC.ActivityCreationDTO(rdf_type="vocabulary:Computation")]
    prov_agent = [

        osC.AgentModel(uri="phis-egi-demo:id/device/hortcontrol_314",
                       rdf_type="vocabulary:Software",
                       settings={}),

        ]

    body = osC.ProvenanceCreationDTO(name=prov,
                                     description=description,
                                     prov_agent=prov_agent,
                                     prov_activity=prov_activity)

    Api_Resp = Dat_Api.create_provenance(body=body, )
    print("Provenance Created: {}".format(str(Api_Resp["metadata"]["datafiles"])))
    Prov_Src = Dat_Api.search_provenance(name=prov,)["result"]
    prov_dict.update({prov: Prov_Src[0].uri})
    print("{} URI Created: {}".format(prov, Prov_Src[0].uri))
    del description, prov_activity, prov_agent, body, Api_Resp
del prov, Dat_Api, Prov_Src

# Import datafiles (Images, Mesh...)



## List all Files Names

## Get links between point cloud file names, timestamps, and plant IDs
<img src='https://drive.google.com/uc?export=view&id=1mhJBdYvRxc8E51d97qs74NraKVrrWsAP'>

We have listed the point cloud file names in our dataset to enable matching between point cloud data and the associated collected data and metadata.

In [ ]:
### Create a new dataframe with unique rows from 'Tiemstamp' to 'Image Name', dropping duplicates and NaN values ############
df_Files = df_data.loc[:, "Block":"Filename"].drop_duplicates().dropna()

# Display the first few rows of the new dataframe
display(df_Files.head())

### Initialize an empty list to store file paths ############################################################################
ls_files = []

### Walk through the directory tree #########################################################################################
for (root, dirs, files) in os.walk(wd):
    # Iterate over each file in the current directory
    for filename in files:
        # Check if the file has a .png extension
        if filename.endswith(".ply"):
            # Add the full file path to the list
            ls_files.append(os.path.join(root, filename))

### Print the number of FishEyeCorrected and FishEyeMasked images ###########################################################
print(f'Number of Files to import: {len(ls_files)}')

### Need to link proper Metadata to Point Cloud
<img src='https://drive.google.com/uc?export=view&id=1AFMLb6mpCn7tPGP09d6JL9rPkoNZ5in0'>

In [ ]:
prov = 'TraitFinder_Mesh'
dic_filepath = {}

for path in ls_files:
    name = os.path.basename(path)

    # Recover timestamp if you want (optional)
    filtered = df_Files.loc[df_Files["Filename"] == name, "Timestamp"]
    timestamp = filtered.iloc[0] if not filtered.empty else None  # optional

    # Build lookup directly
    dic_filepath[name] = {
        "Path": path,
        "Prov": prov_dict[prov],
        "Timestamp": timestamp  # optional, include if needed
    }

# Quick check
#print(list(dic_filepath.items())[:2])

# Safely get Path and Prov from dic_filepath
df_Files["Path"] = df_Files["Filename"].map(lambda x: dic_filepath.get(x, {}).get("Path"))
df_Files["Prov"] = df_Files["Filename"].map(lambda x: dic_filepath.get(x, {}).get("Prov"))

# Map Plant_URI
df_Files["Plant_URI"] = df_Files["Plant_ID"].map(ScObj_uri)

df_missing_path = df_Files[df_Files["Path"].isna()]
print(f'Number of files without Path: {len(df_missing_path)}')

df_files_to_import = df_Files.dropna(subset=["Path"])
display(df_files_to_import.head())
print(f'Number of files with Path: {len(df_files_to_import)}')

In [ ]:
df_files_to_import = df_files_to_import.copy()

df_files_to_import["Mesh URI"] = None

for index, row in tqdm(df_files_to_import.iterrows(), desc="Files processing:"):
  target_uri = row["Plant_URI"]
  img_name = row["Filename"]
  timestamp = row["Timestamp"].replace('+', '.000+')

  Dat_Api = osC.DataApi(Py_Client)
  Dat_Src = Dat_Api.get_data_file_descriptions_by_targets(targets = [target_uri],)["result"]

  matched_uri = None   # remember result

  for elt in Dat_Src:
    if elt.filename == img_name and elt._date == timestamp:
      #print(f'Match Filename: {elt.filename} - {img_name}')
      #print(f'Match Time: {elt._date} - {timestamp}')
      matched_uri = elt.uri   # store it
      df_files_to_import.loc[row.name, 'Mesh URI'] = matched_uri

print('/n')
df_imported_file=df_files_to_import[df_files_to_import['Mesh URI'].notna()]
print(f'Number of files already Imported: {len(df_imported_file)}')
df_files_to_import=df_files_to_import[df_files_to_import['Mesh URI'].isna()]
print(f'Number of files to Import: {len(df_files_to_import)}')

### Import Images
<img src='https://drive.google.com/uc?export=view&id=1uRTbRD8-IEFTSyNIhYL4n5fXLUDXer22'>

In [ ]:
### Refresh Connection to the Database ######################################################################################
# Connect to the OpenSILEX web service using the provided login credentials
Py_Client.connect_to_opensilex_ws(identifier=login["Identifier"],
                                  password=login["Password"],
                                  host=login["Host"])

# Set a time limit of 30 minutes from the current time
timelimit = datetime.datetime.now() + datetime.timedelta(minutes=30)

# Iterate over each image in the Corr_Name list
for index, row in tqdm(df_files_to_import.iterrows(), desc="Files processing:"):
  # Create a description dictionary for the image
  description = {
      "rdf_type": "vocabulary:3DMesh",  # Originally proposed as a DTO object
      "date": row["Timestamp"],
      "target": row["Plant_URI"],
      "metadata": {'Block': row['Block'],
                   'Column': row['Column'],
                   'Row': row['Row']}, ## ADD Plant POSITION
      "provenance": {
          "uri": row["Prov"],
          "settings": {},
          "experiments": [NameExp_uri[NameExp]]
      }
  }
  # Post the image file along with its description to the Data API
  Dat_Api.post_data_file(description=json.dumps(description), file=row["Path"])  # DTO object replaced by json.dumps()
  # Clean up by deleting the image and description variables


    # Reconnect to the Database After 30 Minutes ##############################################################################
  # Check if the current time exceeds the time limit
  if datetime.datetime.now() > timelimit:
      # Reconnect to the OpenSILEX web service using the provided login credentials
      Py_Client.connect_to_opensilex_ws(identifier=login["Identifier"],
                                        password=login["Password"],
                                        host=login["Host"])
      # Reinitialize the Data API with the Py_Client instance
      Dat_Api = osC.DataApi(Py_Client)
      # Reset the time limit to 30 minutes from the current time
      timelimit = datetime.datetime.now() + datetime.timedelta(minutes=30)
      # print(Py_Client.default_headers['Authorization'])

print('Done')

In [ ]:
df_files_to_import = df_files_to_import.copy()

for index, row in tqdm(df_files_to_import.iterrows(), desc="Files processing:"):
  target_uri = row["Plant_URI"]
  img_name = row["Filename"]
  timestamp = row["Timestamp"].replace('+', '.000+')

  Dat_Api = osC.DataApi(Py_Client)
  Dat_Src = Dat_Api.get_data_file_descriptions_by_targets(targets = [target_uri],)["result"]

  matched_uri = None   # remember result

  for elt in Dat_Src:
    if elt.filename == img_name and elt._date == timestamp:
      #print(f'Match Filename: {elt.filename} - {img_name}')
      #print(f'Match Time: {elt._date} - {timestamp}')
      matched_uri = elt.uri   # store it
      df_files_to_import.loc[row.name, 'Mesh URI'] = matched_uri

print('/n')
df_temp=df_files_to_import[df_files_to_import['Mesh URI'].notna()]
df_imported_file=pd.concat([df_temp, df_imported_file])
print(f'Number of files Imported: {len(df_imported_file)}')

df_files_to_import=df_files_to_import[df_files_to_import['Mesh URI'].isna()]
print(f'Number of files to Import: {len(df_files_to_import)}')

In [ ]:
df_imported_file.head()

# Import Numerical Data
<img src='https://drive.google.com/uc?export=view&id=1Y0vid9JzsjaCdCaU3KPrJqYgMwRsps1k'>

## Get link between Columns Names and Variables
This section establishes the mapping between observed variables and their corresponding column names in the dataset.

In [ ]:
### Construct the path to the '00-yaml' directory within the working directory ##############################################
wd_yaml = os.path.join(wd, '00-yaml')

### Open the 'Morpho_Info.yaml' file in read mode and load its contents #####################################################
with open(os.path.join(wd_yaml, 'Morpho_Info.yaml'), 'r') as stream:
    Morpho_Info = yaml.safe_load(stream)
del(wd_yaml)

### Display the contents of the 'Morpho_Info' dictionary ####################################################################
display(Morpho_Info)

## Filter datapoint based existing datafile URI

In [ ]:
df_data = pd.merge(
    df_data,
    df_imported_file.drop(columns=['Prov', 'Path'], errors='ignore'),
    on=['Block', 'Column', 'Row', 'Timestamp', 'Plant_ID', 'Filename'],
    how='left'   # keep all df_data rows
)

df_data_with_uri = df_data[df_data['Mesh URI'].notna()].copy()
#display(df_with_uri.head())
print(f'Number of datapoint link to files: {len(df_data_with_uri)}')

df_data_without_uri = df_data[df_data['Mesh URI'].isna()].copy()
#display(df_without_uri.head())
print(f'Number of datapoint not link to files: {len(df_data_without_uri)}')

## Numerical Data Import
This section handles the import of numerical data from the dataset.

### Import datapoint link to files
<img src='https://drive.google.com/uc?export=view&id=1uuYav1he9J6AEJ5inCg3NfO89u8OIIO9'>

In [ ]:
### Set the Provenance ######################################################################################################
prov = 'HortControl 3.14_Parameters'

### Connect to the OpenSILEX web service using the provided login credentials ###############################################
Py_Client.connect_to_opensilex_ws(identifier=login["Identifier"],
                                  password=login["Password"],
                                  host=login["Host"])

# Set a time limit of 30 minutes from the current time
timelimit = datetime.datetime.now() + datetime.timedelta(minutes=30)

### Data Import #############################################################################################################
# Initialize the Data API and Variables API with the Py_Client instance
Dat_Api = osC.DataApi(Py_Client)
Var_Api = osC.VariablesApi(Py_Client)

# Define the provenance entity model
Prov_Was_Associated_With = osC.ProvEntityModel(uri="phis-egi-demo:id/device/hortcontrol_314",
                                               rdf_type="vocabulary:Software")

# Initialize an empty dictionary to log duplicate data
logfile = {}

### Iterate over each item in the Morpho_Info dictionary ####################################################################
for key, value in tqdm(Morpho_Info.items()):
    logfile[value] = []
    # Search for the variable source using the variable name
    Var_Src = Var_Api.search_variables(name=value)["result"]

    # Get or Create Numerical Data ##########################################################################################
    pas = 1000  # Set the batch size for data import
    count = 0  # Initialize a counter

    # Iterate over the data in slices of size 'pas'
    for slc in range(0, len(df_data), pas):
        df_Slice = df_data_with_uri.iloc[slc:slc + pas]  # Slice the dataframe
        bodies = []  # Initialize an empty list to store data bodies
        count += 1  # Increment the counter

        # Iterate over each row in the sliced dataframe
        for index, row in df_Slice.iterrows():
            # Search for existing data in the Data API
            Dat_Src = Dat_Api.search_data_list(targets=[ScObj_uri[row["Plant_ID"]]],
                                               start_date=row['Timestamp'].replace('+', '.000+'),
                                               end_date=row['Timestamp'].replace('+', '.000+'),
                                               variables=[Var_Src[0].uri],
                                               experiments=[NameExp_uri[NameExp]], page_size=20)['result']
            if Dat_Src:
                # Log the data if it already exists
                logfile[value].append({'Plant_ID': {row["Plant_ID"]},
                                       'Timestamp': {row["Timestamp"]}})
            else:
                Prov_Used = osC.ProvEntityModel(uri=row["Mesh URI"], rdf_type="vocabulary:3DMesh")  # Initialize the provenance used variable
                #Setting_Dict = {"Camera Angle": "{:03d}".format(row["Angle"])}  # Create a settings dictionary

                # Create a data creation DTO object
                body = osC.DataCreationDTO(_date=str(row['Timestamp']),
                                           target=ScObj_uri[row["Plant_ID"]],
                                           variable=Var_Src[0].uri,
                                           value=row[key],
                                           metadata={'Block': row['Block'],
                                                      'Column': row['Column'],
                                                      'Row': row['Row']},
                                           provenance=osC.DataProvenanceModel(
                                               uri=prov_dict[prov],
                                               prov_used=[Prov_Used],
                                               prov_was_associated_with=[Prov_Was_Associated_With],
                                               experiments=[NameExp_uri[NameExp]]))
                bodies.append(body)  # Append the DTO object to the list

                # Reconnect to the OpenSILEX web service if the time limit is exceeded
                if datetime.datetime.now() > timelimit:
                    Py_Client.connect_to_opensilex_ws(identifier=login["Identifier"],
                                                      password=login["Password"],
                                                      host=login["Host"])
                    Dat_Api = osC.DataApi(Py_Client)
                    timelimit = datetime.datetime.now() + datetime.timedelta(minutes=30)

        if bodies:
            # Add the list of data bodies to the Data API
            Dat_Api.add_list_data(body=bodies)
        else:
            print(f'all data of {value} already uploaded')

print('Import Over')

### Import datapoint not link to files
*(2min30s for one Variable)*

In [ ]:
### Set the Provenance ######################################################################################################
prov = 'HortControl 3.14_Parameters'

### Connect to the OpenSILEX web service using the provided login credentials ###############################################
Py_Client.connect_to_opensilex_ws(identifier=login["Identifier"],
                                  password=login["Password"],
                                  host=login["Host"])

# Set a time limit of 30 minutes from the current time
timelimit = datetime.datetime.now() + datetime.timedelta(minutes=30)

### Data Import #############################################################################################################
# Initialize the Data API and Variables API with the Py_Client instance
Dat_Api = osC.DataApi(Py_Client)
Var_Api = osC.VariablesApi(Py_Client)

# Define the provenance entity model
Prov_Was_Associated_With = osC.ProvEntityModel(uri="phis-egi-demo:id/device/hortcontrol_314",
                                               rdf_type="vocabulary:Software")

# Initialize an empty dictionary to log duplicate data
logfile = {}

### Iterate over each item in the Morpho_Info dictionary ####################################################################
for key, value in tqdm(Morpho_Info.items()):
    logfile[value] = []
    # Search for the variable source using the variable name
    Var_Src = Var_Api.search_variables(name=value)["result"]

    # Get or Create Numerical Data ##########################################################################################
    pas = 1000  # Set the batch size for data import
    count = 0  # Initialize a counter

    # Iterate over the data in slices of size 'pas'
    for slc in range(0, len(df_data), pas):
        df_Slice = df_data_without_uri.iloc[slc:slc + pas]  # Slice the dataframe
        bodies = []  # Initialize an empty list to store data bodies
        count += 1  # Increment the counter

        # Iterate over each row in the sliced dataframe
        for index, row in df_Slice.iterrows():
            # Search for existing data in the Data API
            Dat_Src = Dat_Api.search_data_list(targets=[ScObj_uri[row["Plant_ID"]]],
                                               start_date=row['Timestamp'].replace('+', '.000+'),
                                               end_date=row['Timestamp'].replace('+', '.000+'),
                                               variables=[Var_Src[0].uri],
                                               experiments=[NameExp_uri[NameExp]], page_size=20)['result']
            if Dat_Src:
                # Log the data if it already exists
                logfile[value].append({'Plant_ID': {row["Plant_ID"]},
                                       'Timestamp': {row["Timestamp"]}})
            else:
                Prov_Used = None  # Initialize the provenance used variable
                #Setting_Dict = {"Camera Angle": "{:03d}".format(row["Angle"])}  # Create a settings dictionary

                # Create a data creation DTO object
                body = osC.DataCreationDTO(_date=str(row['Timestamp']),
                                           target=ScObj_uri[row["Plant_ID"]],
                                           variable=Var_Src[0].uri,
                                           value=row[key],
                                           metadata={'Block': row['Block'],
                                                      'Column': row['Column'],
                                                      'Row': row['Row']},
                                           provenance=osC.DataProvenanceModel(
                                               uri=prov_dict[prov],
                                               prov_used=[Prov_Used],
                                               prov_was_associated_with=[Prov_Was_Associated_With],
                                               experiments=[NameExp_uri[NameExp]]))
                bodies.append(body)  # Append the DTO object to the list

                # Reconnect to the OpenSILEX web service if the time limit is exceeded
                if datetime.datetime.now() > timelimit:
                    Py_Client.connect_to_opensilex_ws(identifier=login["Identifier"],
                                                      password=login["Password"],
                                                      host=login["Host"])
                    Dat_Api = osC.DataApi(Py_Client)
                    timelimit = datetime.datetime.now() + datetime.timedelta(minutes=30)

        if bodies:
            # Add the list of data bodies to the Data API
            Dat_Api.add_list_data(body=bodies)
            aa=body
        else:
            print(f'all data of {value} already uploaded')

print('Import Over')